In [1]:
# general imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
import seaborn as sns
from sbi.inference import MNLE
from sbi.utils import MultipleIndependent
import torch
from torch.distributions import Uniform
import matplotlib.pylab as pl
import time
from pybads import BADS
import warnings
import os
import logging
import pickle
warnings.filterwarnings('ignore')
PYTENSOR_FLAGS=''
logging.getLogger('matplotlib.font_manager').disabled = True

# append the path of the python scripts
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path+"\\scripts")


c:\Users\wenlou\.conda\envs\cluster\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
import io

#https://stackoverflow.com/questions/57081727/load-pickle-file-obtained-from-gpu-to-cpu
class CPU_Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'torch.storage' and name == '_load_from_bytes':
            return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
        else:
            return super().find_class(module, name)

with open("M:/1confiProj/MNLE/base_model_1_estimator.pickle", "rb") as f:
    estimator = CPU_Unpickler(f).load()


In [ ]:
data_path = 'P:/3026008.02/MNLE'
exp_name = 'base'
m_id =1 

targetname = exp_name + '_model_' + str(m_id)
sim_agg_data_path = os.path.join(data_path, exp_name, 'model_' + str(m_id), 'agg_data')
with open(os.path.join(sim_agg_data_path, targetname + '_agg.pickle'), 'rb') as handle:
    all_sim_dat, all_theta = pickle.load(handle)
num_simulations = all_sim_dat.shape[0]
print('---------data loaded---------')

---------data loaded---------


In [6]:
all_sim_dat[:,-2] = all_sim_dat[:,-2]/100.0
all_sim_dat

tensor([[ -7.5007,   2.0405,   0.7400,   1.0000],
        [-11.1510,   4.4634,   0.3600,   1.0000],
        [-11.3723,   0.3233,   0.9300,   1.0000],
        ...,
        [ 17.6142,   8.0055,   1.0000,   1.0000],
        [  8.3437,  15.1148,   1.0000,   1.0000],
        [  2.8766,   0.9810,   0.9500,   0.0000]])

In [5]:
all_theta

tensor([[  4.8991,   4.6623,  13.2262,  ..., -10.0000,   1.0000,   1.0000],
        [  1.7309,   0.2693,   4.4808,  ..., -10.0000,   1.0000,   1.0000],
        [  1.7066,   2.6029,   0.8898,  ..., -10.0000,   1.0000,   1.0000],
        ...,
        [  4.4364,   4.1462,  13.3882,  ...,  10.0000,   2.0000,   0.0000],
        [  3.6982,   7.6858,   9.5916,  ...,  10.0000,   2.0000,   0.0000],
        [  4.9114,   3.4920,   4.8596,  ...,  10.0000,   2.0000,   0.0000]])

In [7]:
theta = all_theta[0,:]
theta = torch.reshape(torch.tensor(theta), (1, len(theta))).to(torch.float32)
theta = theta.repeat(2, 1)


In [12]:
all_theta.shape

torch.Size([2260000, 15])

In [10]:
# define function to be minimized by BADS (negative log-likelihood)
# The conditioned_potential_fn expects theta and x_o (full observed data).
# We need to ensure x_o has the correct shape for the estimator's log_prob: (sample_dim, batch_dim, *event_shape)
# sample_dim = 1, batch_dim = num_trials, event_shape = 2 (RT, choice)
# data_torch[:, nConditions:] gives us the behavioral data with shape (num_trials, 2)
# We need to reshape it to (1, num_trials, 2)
eps=1e-3
CTE = 1/2 * 1/600 * 1/995
x_o = all_sim_dat[:2, :].reshape(1, 2, all_sim_dat.shape[1])
log_liks = estimator.log_prob(x_o, condition=theta).detach().numpy()  #take log prob from MNLE
log_liks = np.exp(log_liks)*(1-eps) + eps*CTE  # add contaminants
log_liks = np.log(log_liks)  # take log prob
log_liks_no_fb = -np.nansum(log_liks)  # sum over all data points to get -LL (data | theta) for RT > 0

In [ ]:
def compute_NLL(theta, estimator, data, CTE, eps=1e-3):
    '''
    inputs:
    theta: np.array() parameters at which to evaluate the NLL
    data: beh data (including trial conditions)
    '''
    ntrials = data.shape[0] # number of trials
    cond = data[:, :4]
    theta = torch.reshape(torch.tensor(theta), (1, len(theta))).to(torch.float32)
    theta = theta.repeat(ntrials, 1)
    theta = torch.column_stack((theta, torch.tensor(cond).to(torch.float32)))
    #reorg data
    x_o = data[:, 4:]
    x_o = torch.reshape(torch.tensor(x_o), (1, ntrials, 4)).to(torch.float32)
    log_liks = estimator.log_prob(x_o, condition=theta).detach().numpy()  #take log prob from MNLE
    log_liks = np.exp(log_liks)*(1-eps) + eps*CTE  # add contaminants
    log_liks = np.log(log_liks)  # take log prob

    return -np.nansum(log_liks)

def get_beh_data_MNLE(sub_id):
    data_path = 'P:/3026008.02/data_for_fitting/'
    sub_data = np.loadtxt(os.path.join(data_path, 'beh_data_sub_' + str(sub_id) + '.csv'), delimiter=',')

    # reorg data to be consistent with the training input
    sub_data = sub_data[:, [0, 1, 2, 3, 4, 5, 7, 6]] # the catrgorical condition is the last column
    sub_data[:, -1] = 2 - sub_data[:, -1] # convert the 1 - 2 choice to 0 - 1 choice
    sub_data[:,-2] = sub_data[:,-2]/100.0

    return sub_data


In [ ]:
data = get_beh_data_MNLE(2)
theta =np.array([4.8991,   4.6623,  13.2262,  18.1474,  -4.5114,  23.3149,   0.6695,
          0.0418,   0.2384,   0.2031,   0.7015])
compute_NLL(theta, estimator, data, CTE, eps=1e-3)

np.float32(32426.664)

In [28]:
-np.nansum(log_liks)

np.float32(32426.664)